In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchview import draw_graph

In [52]:
# -------- Basic Conv Block --------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dp=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(),
            nn.Dropout3d(dp),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(),
            )

    def forward(self, x):
        return self.block(x)

# -------- Decoder Block --------
class DecoderBlock3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dp=0.1):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, (1, 2, 2), stride=(1, 2, 2))
        self.conv_block = ConvBlock(out_channels + skip_channels, out_channels, dp)
        
    def forward(self, x, skip):
        x = self.up(x)

        # Handle size mismatches
        x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)

        x = self.conv_block(x)
        return x

# -------- Temporal Aggregation Block --------
class TemporalAggregationBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.temporal_attn = nn.Sequential(
            nn.Conv3d(in_channels, in_channels, kernel_size=(3, 1, 1), padding=(1, 0, 0), groups=in_channels),
            nn.Sigmoid()
        )
        self.temporal_score = nn.Conv3d(in_channels, 1, kernel_size=1)
    
    def forward(self, x):
        # x shape: (B, C, T, H, W)
        attn = self.temporal_attn(x)
        feat = x * attn
        score = self.temporal_score(feat)
        weights = torch.softmax(score, dim=2) 
        feat = (feat * weights).sum(dim=2)    
        return feat

# -------- Global Attention Branch --------
class GlobalAttentionBlock(nn.Module):
    def __init__(self, channels, time_pooling=None):
        super().__init__()
        reduction = max(channels // 4, 8)
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool3d((time_pooling, 1, 1)),
            nn.Conv3d(channels, reduction, 1),
            nn.PReLU(reduction),
            nn.Conv3d(reduction, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * (1+self.branch(x))
    
    
class GatedTemporalMixBlock(nn.Module):
    def __init__(self, channels, window_size=5):
        super().__init__()

        if window_size % 2 == 0:
            raise ValueError("window_size must be odd")

        pad_t = window_size // 2
        kernel = (window_size, 1, 1)
        padding = (pad_t, 0, 0)

        self.value = nn.Sequential(
            nn.Conv3d(
                channels,
                channels,
                kernel_size=kernel,
                padding=padding,
                bias=False,
            ),
            nn.BatchNorm3d(channels),
            nn.PReLU(channels),
        )

        self.gate = nn.Sequential(
            nn.Conv3d(
                channels,
                channels,
                kernel_size=kernel,
                padding=padding,
            ),
            nn.Sigmoid(),
        )

    def forward(self, x):
        mix = self.value(x)
        gate = self.gate(x)
        return x + (mix * gate)
    
class RegressionHead(nn.Module):

    def __init__(self, base_ch: int, out_channels: int, out_hw: tuple):
        
        super(RegressionHead, self).__init__()
        self.out_hw = out_hw
        self.refinement= nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.PReLU(base_ch),
        )
        
        self.regressor = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.PReLU(base_ch),
            nn.Dropout2d(0.1),
            nn.Conv2d(base_ch, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = self.refinement(x)

        # Interpolate spatial dimensions to match out_hw
        x = F.interpolate(x, size=self.out_hw, mode="bilinear", align_corners=False)
        
        # Apply regression layers
        out = self.regressor(x)

        return out

class ClassificationHead(nn.Module):

    def __init__(self, base_ch: int, out_channels: int, out_hw: tuple):
        
        super(ClassificationHead, self).__init__()
        self.out_hw = out_hw
        self.refinement= nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.PReLU(base_ch),
        )
        
        self.regressor = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.PReLU(base_ch),
            nn.Conv2d(base_ch, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = self.refinement(x)

        # Interpolate spatial dimensions to match out_hw
        x = F.interpolate(x, size=self.out_hw, mode="bilinear", align_corners=False)
        
        # Apply regression layers
        out = self.regressor(x)
        return out

In [53]:
B, T, C, H, W = 1, 12, 21, 65, 25
H_out, W_out = 30, 17

In [54]:

block = ConvBlock(
    in_ch=C,
    out_ch=C*2,
    dp=0.2
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, C, T, H, W),
    expand_nested=True,
    roll=True,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\ConvBlock.png'

In [55]:
block = GlobalAttentionBlock(
    channels = C
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, C, T, H, W),
    expand_nested=True,
    roll=False,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\GlobalAttentionBlock.png'

In [56]:
block = GatedTemporalMixBlock(
    channels = C,
    window_size = 7
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, C, T, H, W),
    expand_nested=True,
    roll=False,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\GatedTemporalMixBlock.png'

In [57]:
block = TemporalAggregationBlock(
    in_channels = 7,
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, 7, T, H, W),
    expand_nested=True,
    roll=False,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\TemporalAggregationBlock.png'

In [58]:
block = RegressionHead(
    base_ch=7,
    out_channels=1,
    out_hw=(H_out, W_out),
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, 7, H, W),
    expand_nested=True,
    roll=False,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\RegressionHead.png'

In [59]:
block = ClassificationHead(
    base_ch=7,
    out_channels=3,
    out_hw=(H_out, W_out),
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=(B, 7, H, W),
    expand_nested=True,
    roll=False,
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\ClassificationHead.png'

In [65]:
block = DecoderBlock3D(
    in_channels=14,
    skip_channels=7,
    out_channels=7,
    dp=0.2
)

name = block.__class__.__name__

graph = draw_graph(
    block,
    input_size=[(B, 14, T, H//2, W//2), (B, 7, T, H, W)],
    expand_nested=True,
    roll=False,
    depth=1
)

graph.visual_graph.graph_attr.update(dpi="200")
graph.visual_graph.render(f"./modelos/diagramas_bloques/{name}", format="png")

couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.

'modelos\\diagramas_bloques\\DecoderBlock3D.png'